# Estrous Detection from Ultradian Activity Rhythms (Positive Control) — Per-Mouse GMM

This notebook replicates the Morph2REP-style positive control pipeline on activity-only data:
female locomotor activity (1-minute resolution, 14 days, 13 mice).

**Goals**
1. QC daily coverage (Morph2REP analog).
2. Compute wavelet-band features, focusing on **ultradian power (1–3 hours)**.
3. Validate estrus effect using known estrus timing:
   - Professor-confirmed estrus day: **day==1** (0-based indexing).
   - Paper-like periodic days: **{1,5,9,13}** (0-based).
4. Derive an interpretable **estrus-likeness probability** via a **per-mouse 1D GMM** on `z_U_1_3h`.
5. Demonstrate **label-free 4-day cyclicity** using a **lag-4 permutation test** on `z_U_1_3h`.

> Note: We intentionally do **not** use temperature; this notebook validates estrus signal using **activity only**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score

import pywt
import time

plt.rcParams["figure.dpi"] = 120
np.set_printoptions(precision=4, suppress=True)

## 1) Load data (Fem Act) and reshape to long format

`Fem Act` is a 20160×13 matrix (14 days × 1440 minutes/day) for female mice `f1..f13`.

We construct:
- `minute`: global minute index 0..20159  
- `day`: 0..13  
- `minute_of_day`: 0..1439  
- `act_long`: tidy table with columns `mouse_id, minute, day, minute_of_day, act`

In [ ]:
PATH = "Mouse_Data_Student_Copy.xlsx"

act_wide = pd.read_excel(PATH, sheet_name="Fem Act")
assert act_wide.shape == (20160, 13)
assert all(str(c).startswith("f") for c in act_wide.columns)

act = act_wide.copy()
act["minute"] = np.arange(len(act))
act["day"] = act["minute"] // 1440
act["minute_of_day"] = act["minute"] % 1440

act_long = (act.melt(
    id_vars=["minute","day","minute_of_day"],
    value_vars=list(act_wide.columns),
    var_name="mouse_id",
    value_name="act"
).sort_values(["mouse_id","minute"]).reset_index(drop=True))

print("act_long shape:", act_long.shape)
display(act_long.head())

## 2) QC: remove low-coverage days (Morph2REP analog)

We use **minutes with activity > 0** per day:
- `minutes_act_gt0 = count(act > 0)` per mouse-day  
- Keep days with `MINUTES_FLOOR <= minutes_act_gt0 <= MINUTES_CAP`

We set:
- `MINUTES_FLOOR = 600`
- `MINUTES_CAP = 1300` (conservative upper cap)

In [ ]:
MINUTES_FLOOR = 600
MINUTES_CAP   = 1300

qc_daily = (act_long
    .assign(present=(act_long["act"] > 0).astype(int))
    .groupby(["mouse_id","day"])
    .agg(minutes_act_gt0=("present","sum"),
         act_mean=("act","mean"),
         act_std=("act","std"),
         act_max=("act","max"))
    .reset_index()
)

qc_daily["qc_keep"] = (qc_daily["minutes_act_gt0"] >= MINUTES_FLOOR) & (qc_daily["minutes_act_gt0"] <= MINUTES_CAP)

print("QC keep rate:", float(qc_daily["qc_keep"].mean()))
print("Days removed per mouse:")
display(qc_daily.groupby("mouse_id")["qc_keep"].apply(lambda s: int((~s).sum())).sort_values(ascending=False))

plt.figure()
plt.hist(qc_daily["minutes_act_gt0"], bins=30)
plt.axvline(MINUTES_FLOOR, linestyle="--")
plt.title("Daily activity coverage (minutes with act > 0)")
plt.xlabel("minutes_act_gt0")
plt.ylabel("count (mouse-days)")
plt.show()

## 3) Wavelet-derived features (ultradian + circadian band power)

Per mouse:
- `U_t`: per-minute **ultradian band (1–3h) max wavelet power**
- `C_t`: per-minute **circadian band (23–25h) max wavelet power**

Daily summaries:
- `U_1_3h = mean(U_t within day)`
- `C_23_25h = mean(C_t within day)`
- `log_U_over_C = log(U/C)`

We compute only two bands to keep runtime reasonable.

In [ ]:
DT = 60.0
WAVELET = "cmor1.5-1.0"
fc = pywt.central_frequency(WAVELET)

def scales_for_period_hours(period_hours: float) -> float:
    freq_hz = 1.0 / (period_hours * 3600.0)
    return fc / (freq_hz * DT)

n_scales_per_band = 24
periods_ultr = np.linspace(1.0, 3.0, n_scales_per_band)
periods_circ = np.linspace(23.0, 25.0, n_scales_per_band)

scales_ultr = np.array([scales_for_period_hours(p) for p in periods_ultr], dtype=float)
scales_circ = np.array([scales_for_period_hours(p) for p in periods_circ], dtype=float)

def wavelet_band_series_bandonly(x_14days: np.ndarray):
    x = np.asarray(x_14days, dtype=np.float32)
    coef_u, _ = pywt.cwt(x, scales_ultr, WAVELET, sampling_period=DT)
    coef_c, _ = pywt.cwt(x, scales_circ, WAVELET, sampling_period=DT)
    U_t = (np.abs(coef_u) ** 2).max(axis=0)
    C_t = (np.abs(coef_c) ** 2).max(axis=0)
    return U_t, C_t

t0 = time.time()
rows = []
mice = sorted(act_long["mouse_id"].unique())

for i, mouse_id in enumerate(mice, start=1):
    df_m = act_long[act_long["mouse_id"] == mouse_id].sort_values("minute")
    x = df_m["act"].to_numpy()
    U_t, C_t = wavelet_band_series_bandonly(x)

    for day in range(14):
        lo, hi = day * 1440, (day + 1) * 1440
        rows.append((mouse_id, day, float(np.mean(U_t[lo:hi])), float(np.mean(C_t[lo:hi]))))

    if i % 3 == 0 or i == len(mice):
        print(f"[{i}/{len(mice)}] done {mouse_id}  (elapsed {time.time() - t0:.1f}s)")

feat = pd.DataFrame(rows, columns=["mouse_id", "day", "U_1_3h", "C_23_25h"])
feat["log_U_over_C"] = np.log((feat["U_1_3h"] + 1e-12) / (feat["C_23_25h"] + 1e-12))

feat = feat.merge(qc_daily[["mouse_id","day","minutes_act_gt0","qc_keep"]], on=["mouse_id","day"], how="left")
feat_qc = feat.loc[feat["qc_keep"]].copy()

print("feat_qc shape:", feat_qc.shape)
display(feat_qc.head())

## 4) Robust z-scoring per mouse

`z = (x - median) / (1.4826 * MAD)`

In [ ]:
def robust_z(x):
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-12
    return (x - med) / (1.4826 * mad)

for col in ["U_1_3h", "C_23_25h", "log_U_over_C"]:
    feat_qc[f"z_{col}"] = feat_qc.groupby("mouse_id")[col].transform(robust_z)

display(feat_qc[["mouse_id","day","minutes_act_gt0","z_U_1_3h","z_C_23_25h","z_log_U_over_C"]].head())

### Figure: heatmap of ultradian power across days

In [ ]:
pivot = feat_qc.pivot(index="mouse_id", columns="day", values="z_U_1_3h").sort_index()

plt.figure(figsize=(9, 3.2))
im = plt.imshow(pivot.values, aspect="auto")
plt.title("Ultradian power across days (z_U_1_3h)")
plt.xlabel("day (0-based)")
plt.ylabel("mouse")
plt.colorbar(im, label="z_U_1_3h")
plt.xticks(range(pivot.shape[1]), list(pivot.columns))
plt.yticks(range(pivot.shape[0]), list(pivot.index))
plt.show()

## 5) Estrus validation using known estrus day(s)

In [ ]:
df = feat_qc.copy()
df["is_estrus_A"] = (df["day"] == 1)
df["is_estrus_B"] = df["day"].isin([1,5,9,13])

def within_mouse_effect(df, estrus_col, value_col="z_U_1_3h"):
    rows = []
    for mid, g in df.groupby("mouse_id"):
        e = g.loc[g[estrus_col], value_col]
        n = g.loc[~g[estrus_col], value_col]
        if len(e) == 0 or len(n) == 0:
            continue
        rows.append((mid, float(e.mean()), float(n.mean()), float(e.mean() - n.mean())))
    return pd.DataFrame(rows, columns=["mouse_id","estrus_mean","non_mean","diff_estrus_minus_non"])

for label in ["is_estrus_A", "is_estrus_B"]:
    eff = within_mouse_effect(df, label, "z_U_1_3h")
    stat, p = wilcoxon(eff["diff_estrus_minus_non"], alternative="less")
    print("\n", label, "n_mice:", len(eff))
    print(eff["diff_estrus_minus_non"].describe())
    print("Wilcoxon p (estrus lower):", p)

plt.figure()
vals0 = df.loc[~df["is_estrus_A"], "z_U_1_3h"].to_numpy()
vals1 = df.loc[df["is_estrus_A"], "z_U_1_3h"].to_numpy()
plt.boxplot([vals0, vals1], labels=["non-estrus", "estrus (day==1)"])
plt.title("Ultradian power is lower on estrus day (activity-only)")
plt.ylabel("z_U_1_3h")
plt.show()

## 6) Per-mouse 1D GMM estrus-likeness score: `p_lowU_1d_mouse`

We fit a **separate 2-component GMM per mouse** on that mouse's `z_U_1_3h` values (QC days).

To guard against unstable fits (small n, component collapse), we compute diagnostics:
- `n_days`: # QC days for mouse
- `sep`: separation = |μ_hi - μ_lo| / sqrt(σ_hi² + σ_lo²)
- `collapsed`: any component weight < `min_weight`

We still compute a probability even if the fit is weak, but we **flag** these mice so you can
interpret results carefully.

In [ ]:
MIN_DAYS_PER_MOUSE = 8       # below this, GMM likely unstable
MIN_WEIGHT = 0.10            # flag if a component weight is too tiny
MIN_SEP = 0.30               # flag if components overlap heavily (heuristic)

def fit_gmm_1d(x, random_state=0):
    X = np.asarray(x, dtype=float).reshape(-1, 1)
    gmm = GaussianMixture(n_components=2, random_state=random_state)
    gmm.fit(X)
    post = gmm.predict_proba(X)
    means = gmm.means_.flatten()
    vars_ = gmm.covariances_.flatten()  # 1D variances
    weights = gmm.weights_.flatten()
    lo = int(np.argmin(means))
    hi = 1 - lo
    # separation heuristic
    sep = float(np.abs(means[hi] - means[lo]) / np.sqrt(vars_[hi] + vars_[lo] + 1e-12))
    collapsed = bool((weights.min() < MIN_WEIGHT))
    return gmm, post[:, lo], means, vars_, weights, lo, sep, collapsed

dfp = feat_qc.copy()
dfp["p_lowU_1d_mouse"] = np.nan

diag_rows = []
for mid, g in dfp.groupby("mouse_id"):
    x = g["z_U_1_3h"].to_numpy()
    n_days = len(x)

    ok_n = (n_days >= MIN_DAYS_PER_MOUSE) and (np.std(x) > 1e-12)

    if not ok_n:
        diag_rows.append({
            "mouse_id": mid,
            "n_days": n_days,
            "means_lo": np.nan, "means_hi": np.nan,
            "w_lo": np.nan, "w_hi": np.nan,
            "var_lo": np.nan, "var_hi": np.nan,
            "sep": np.nan,
            "collapsed": True,
            "ok_fit": False,
            "reason": "too_few_days_or_constant"
        })
        continue

    gmm, p_lo, means, vars_, weights, lo, sep, collapsed = fit_gmm_1d(x, random_state=0)

    dfp.loc[dfp["mouse_id"] == mid, "p_lowU_1d_mouse"] = p_lo

    hi = 1 - lo
    ok_fit = (not collapsed) and (sep >= MIN_SEP)

    diag_rows.append({
        "mouse_id": mid,
        "n_days": n_days,
        "means_lo": float(means[lo]),
        "means_hi": float(means[hi]),
        "w_lo": float(weights[lo]),
        "w_hi": float(weights[hi]),
        "var_lo": float(vars_[lo]),
        "var_hi": float(vars_[hi]),
        "sep": float(sep),
        "collapsed": bool(collapsed),
        "ok_fit": bool(ok_fit),
        "reason": "ok" if ok_fit else ("collapsed" if collapsed else "low_sep")
    })

diag = pd.DataFrame(diag_rows).sort_values("mouse_id").reset_index(drop=True)
print("Per-mouse GMM diagnostics:")
display(diag)

print("\nFraction ok_fit:", float(diag["ok_fit"].mean()))
print("Missing p_lowU_1d_mouse rows:", int(dfp["p_lowU_1d_mouse"].isna().sum()))

### Figure: per-mouse GMM parameters (means and weights)

In [ ]:
plt.figure(figsize=(9, 3.5))
xpos = np.arange(len(diag))
plt.scatter(xpos, diag["means_lo"], label="mean_lo (estrus-like)")
plt.scatter(xpos, diag["means_hi"], label="mean_hi (non-estrus-like)")
plt.xticks(xpos, diag["mouse_id"], rotation=0)
plt.axhline(0, linestyle="--", alpha=0.4)
plt.title("Per-mouse GMM component means on z_U_1_3h")
plt.xlabel("mouse")
plt.ylabel("component mean")
plt.legend()
plt.show()

plt.figure(figsize=(9, 3.5))
plt.scatter(xpos, diag["w_lo"], label="weight_lo")
plt.scatter(xpos, diag["w_hi"], label="weight_hi")
plt.xticks(xpos, diag["mouse_id"], rotation=0)
plt.title("Per-mouse GMM component weights")
plt.xlabel("mouse")
plt.ylabel("component weight")
plt.legend()
plt.show()

### Figure: distribution of `p_lowU_1d_mouse`

In [ ]:
plt.figure()
plt.hist(dfp["p_lowU_1d_mouse"].dropna(), bins=30)
plt.title("Per-mouse p_lowU_1d_mouse distribution")
plt.xlabel("p_lowU_1d_mouse")
plt.ylabel("count")
plt.show()

## 6b) Discrimination of per-mouse `p_lowU_1d_mouse` vs known estrus timing

We compute pooled AUC over all mouse-days (note: probability is mouse-calibrated, not globally calibrated).

In [ ]:
yA = (dfp["day"] == 1).astype(int)
yB = (dfp["day"].isin([1,5,9,13])).astype(int)

mask = dfp["p_lowU_1d_mouse"].notna()
aucA_mouse = roc_auc_score(yA[mask], dfp.loc[mask, "p_lowU_1d_mouse"])
aucB_mouse = roc_auc_score(yB[mask], dfp.loc[mask, "p_lowU_1d_mouse"])

print("AUC p_lowU_1d_mouse vs day==1:", aucA_mouse)
print("AUC p_lowU_1d_mouse vs {1,5,9,13}:", aucB_mouse)

plt.figure()
plt.boxplot(
    [dfp.loc[yA==0, "p_lowU_1d_mouse"].dropna(), dfp.loc[yA==1, "p_lowU_1d_mouse"].dropna()],
    labels=["non-estrus", "estrus (day==1)"]
)
plt.title("Per-mouse GMM: p_lowU higher on estrus day")
plt.ylabel("p_lowU_1d_mouse")
plt.show()

## 7) Label-free 4-day cyclicity detection (Permutation test)

We test whether `z_U_1_3h` shows lag-4 autocorrelation beyond chance.

In [ ]:
def lag4_autocorr_per_mouse(values_by_day: pd.Series):
    y = values_by_day.sort_index().to_numpy(dtype=float)
    if len(y) <= 4:
        return np.nan
    y0, y4 = y[:-4], y[4:]
    if np.std(y0) < 1e-12 or np.std(y4) < 1e-12:
        return np.nan
    return float(np.corrcoef(y0, y4)[0,1])

def cyclicity_score(df, value_col="z_U_1_3h"):
    per = []
    for mid, g in df.groupby("mouse_id"):
        s = g.set_index("day")[value_col]
        per.append(lag4_autocorr_per_mouse(s))
    per = np.array(per, dtype=float)
    per = per[np.isfinite(per)]
    return float(np.mean(per)) if len(per) else np.nan

def monte_carlo_null(df, value_col="z_U_1_3h", n_perm=5000, seed=0):
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        dfp2 = df.copy()
        dfp2[value_col] = dfp2.groupby("mouse_id")[value_col].transform(lambda x: rng.permutation(x.to_numpy()))
        null.append(cyclicity_score(dfp2, value_col=value_col))
    null = np.array(null, dtype=float)
    return null[np.isfinite(null)]

obs = cyclicity_score(feat_qc, value_col="z_U_1_3h")
null = monte_carlo_null(feat_qc, value_col="z_U_1_3h", n_perm=5000, seed=0)
p_mc = (np.sum(null >= obs) + 1) / (len(null) + 1)

print("Observed lag-4 cyclicity (z_U_1_3h):", obs)
print("Monte Carlo p-value:", p_mc)

plt.figure()
plt.hist(null, bins=40)
plt.axvline(obs, linestyle="--")
plt.title("Permutation null for lag-4 cyclicity (z_U_1_3h)")
plt.xlabel("mean lag-4 autocorr across mice")
plt.ylabel("count")
plt.show()

mean_day = feat_qc.groupby("day")["z_U_1_3h"].mean().reindex(range(14))
plt.figure()
plt.plot(mean_day.index, mean_day.values, marker="o")
plt.title("Mean ultradian power across days (z_U_1_3h)")
plt.xlabel("day (0-based)")
plt.ylabel("mean z_U_1_3h")
plt.show()

## 8) Bootstrap uncertainty (cluster bootstrap over mice)

We bootstrap over mice (with replacement) and **preserve duplicates** by concatenation.
This fixes the common `.isin(samp)` pitfall that drops duplicates.

In [ ]:
rng = np.random.default_rng(0)

def bootstrap_by_mouse_df(df, mice_sample):
    parts = []
    for m in mice_sample:
        parts.append(df[df["mouse_id"] == m])
    return pd.concat(parts, axis=0, ignore_index=True)

def bootstrap_auc_cluster(feat_df, label_days, score_col, B=5000, seed=0):
    rng = np.random.default_rng(seed)
    mice = np.array(sorted(feat_df["mouse_id"].unique()))
    aucs = []
    for _ in range(B):
        samp = rng.choice(mice, size=len(mice), replace=True)
        dfb = bootstrap_by_mouse_df(feat_df, samp)
        y = dfb["day"].isin(label_days).astype(int).to_numpy()
        s = dfb[score_col].to_numpy()
        if y.min() == y.max():
            continue
        aucs.append(roc_auc_score(y, s))
    aucs = np.array(aucs, dtype=float)
    return float(np.mean(aucs)), np.quantile(aucs, [0.025, 0.975]), aucs

def per_mouse_diff(df, label_days, value_col="z_U_1_3h"):
    diffs = []
    for mid, g in df.groupby("mouse_id"):
        e = g[g["day"].isin(label_days)][value_col]
        n = g[~g["day"].isin(label_days)][value_col]
        if len(e) == 0 or len(n) == 0:
            continue
        diffs.append(float(e.mean() - n.mean()))
    return np.array(diffs, dtype=float)

def bootstrap_mean_ci(x, B=5000, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    boots = [np.mean(rng.choice(x, size=len(x), replace=True)) for _ in range(B)]
    boots = np.array(boots, dtype=float)
    return float(boots.mean()), np.quantile(boots, [0.025, 0.975]), boots

label_A = [1]
label_B = [1,5,9,13]

# AUC bootstraps for per-mouse probability
mean_auc_A, ci_auc_A, aucs_A = bootstrap_auc_cluster(dfp.dropna(subset=["p_lowU_1d_mouse"]), label_A, "p_lowU_1d_mouse", B=5000, seed=0)
mean_auc_B, ci_auc_B, aucs_B = bootstrap_auc_cluster(dfp.dropna(subset=["p_lowU_1d_mouse"]), label_B, "p_lowU_1d_mouse", B=5000, seed=1)

print("Bootstrap AUC (day==1):", mean_auc_A, "95% CI:", ci_auc_A)
print("Bootstrap AUC (1,5,9,13):", mean_auc_B, "95% CI:", ci_auc_B)

plt.figure()
plt.hist(aucs_A, bins=40)
plt.axvline(0.5, linestyle="--")
plt.axvline(mean_auc_A, linestyle="--")
plt.axvline(ci_auc_A[0], linestyle="--")
plt.axvline(ci_auc_A[1], linestyle="--")
plt.title("Bootstrap AUC distribution (per-mouse p_lowU, day==1)")
plt.xlabel("AUC")
plt.ylabel("count")
plt.show()

# Effect size bootstrap on z_U_1_3h
diffs = per_mouse_diff(feat_qc, label_A, value_col="z_U_1_3h")
boot_mean, ci_diff, boot_d = bootstrap_mean_ci(diffs, B=5000, seed=2)

plt.figure()
plt.hist(boot_d, bins=40)
plt.axvline(0, linestyle="--")
plt.axvline(boot_mean, linestyle="--")
plt.axvline(ci_diff[0], linestyle="--")
plt.axvline(ci_diff[1], linestyle="--")
plt.title("Bootstrap distribution of mean effect (estrus − non) on z_U_1_3h")
plt.xlabel("mean Δ z_U_1_3h")
plt.ylabel("count")
plt.show()

print("Mean diff (estrus - non), day==1:", float(diffs.mean()))
print("Bootstrap mean:", boot_mean, "95% CI:", ci_diff)

## 9) (Optional) PCA visualization of wavelet-derived features

PCA is for qualitative visualization only (not required for inference).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

dfviz = dfp.copy()
dfviz["is_estrus_A"] = (dfviz["day"] == 1)
dfviz["is_estrus_B"] = dfviz["day"].isin([1,5,9,13])

X = dfviz[["z_U_1_3h", "z_C_23_25h", "z_log_U_over_C"]].to_numpy()
X = StandardScaler().fit_transform(X)

pcs = PCA(n_components=2, random_state=0).fit_transform(X)
dfviz["PC1"], dfviz["PC2"] = pcs[:,0], pcs[:,1]

fig, axs = plt.subplots(1, 2, figsize=(10, 4), dpi=120)

axs[0].scatter(dfviz.loc[~dfviz["is_estrus_A"], "PC1"], dfviz.loc[~dfviz["is_estrus_A"], "PC2"], alpha=0.5, label="non-estrus")
axs[0].scatter(dfviz.loc[dfviz["is_estrus_A"], "PC1"], dfviz.loc[dfviz["is_estrus_A"], "PC2"], alpha=0.9, label="estrus (day==1)")
axs[0].set_title("PCA colored by estrus (day==1)")
axs[0].set_xlabel("PC1"); axs[0].set_ylabel("PC2")
axs[0].legend()

axs[1].scatter(dfviz.loc[~dfviz["is_estrus_B"], "PC1"], dfviz.loc[~dfviz["is_estrus_B"], "PC2"], alpha=0.5, label="non-estrus")
axs[1].scatter(dfviz.loc[dfviz["is_estrus_B"], "PC1"], dfviz.loc[dfviz["is_estrus_B"], "PC2"], alpha=0.9, label="estrus (1,5,9,13)")
axs[1].set_title("PCA colored by estrus (4-day cycle)")
axs[1].set_xlabel("PC1"); axs[1].set_ylabel("PC2")
axs[1].legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(6,5), dpi=120)
sc = plt.scatter(dfviz["PC1"], dfviz["PC2"], c=dfviz["p_lowU_1d_mouse"], alpha=0.85)
plt.colorbar(sc, label="p_lowU_1d_mouse  (per-mouse P(low-ultradian))")
plt.title("PCA colored by per-mouse p_lowU_1d_mouse")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

## 10) FINAL SUMMARY CELL (paste this output back to me)

This prints:
- QC retention
- Estrus suppression stats on `z_U_1_3h`
- AUC of per-mouse `p_lowU_1d_mouse` for both label sets
- Per-mouse GMM diagnostics summary (how many unstable fits)
- Lag-4 cyclicity + Monte Carlo p-value

Copy the printed output and paste it to ChatGPT for interpretation.

In [ ]:
print("="*100)
print("POSITIVE CONTROL SUMMARY — PER-MOUSE GMM (Fem Act only)")
print("="*100)

mice = sorted(feat_qc["mouse_id"].unique())
n_mice = len(mice)
n_days_total = n_mice * 14
n_days_kept = len(feat_qc)
qc_keep_rate = n_days_kept / n_days_total

print("\nQC retention:")
print(f"  n_mice = {n_mice}")
print(f"  days_total = {n_days_total} | days_kept = {n_days_kept} | keep_rate = {qc_keep_rate:.3f}")
print(f"  minutes_act_gt0: mean={feat_qc['minutes_act_gt0'].mean():.1f}, min={feat_qc['minutes_act_gt0'].min()}, max={feat_qc['minutes_act_gt0'].max()}")

# Estrus suppression on z_U_1_3h
label_A = [1]
label_B = [1,5,9,13]

diff_A = per_mouse_diff(feat_qc, label_A, value_col="z_U_1_3h")
diff_B = per_mouse_diff(feat_qc, label_B, value_col="z_U_1_3h")

pA = wilcoxon(diff_A, alternative="less").pvalue
pB = wilcoxon(diff_B, alternative="less").pvalue

boot_diffA_mean, boot_diffA_ci, _ = bootstrap_mean_ci(diff_A, B=5000, seed=10)
boot_diffB_mean, boot_diffB_ci, _ = bootstrap_mean_ci(diff_B, B=5000, seed=11)

print("\nEstrus suppression (within-mouse z_U_1_3h):")
print(f"  Label A (day==1): mean diff (estrus-non) = {diff_A.mean():.3f} | Wilcoxon p={pA:.4g}")
print(f"    bootstrap mean={boot_diffA_mean:.3f} 95% CI=[{boot_diffA_ci[0]:.3f}, {boot_diffA_ci[1]:.3f}]")
print(f"  Label B (days 1,5,9,13): mean diff = {diff_B.mean():.3f} | Wilcoxon p={pB:.4g}")
print(f"    bootstrap mean={boot_diffB_mean:.3f} 95% CI=[{boot_diffB_ci[0]:.3f}, {boot_diffB_ci[1]:.3f}]")

# AUC for per-mouse probability
mask = dfp["p_lowU_1d_mouse"].notna()
yA = dfp.loc[mask, "day"].isin(label_A).astype(int).to_numpy()
yB = dfp.loc[mask, "day"].isin(label_B).astype(int).to_numpy()
s  = dfp.loc[mask, "p_lowU_1d_mouse"].to_numpy()

aucA = roc_auc_score(yA, s)
aucB = roc_auc_score(yB, s)

boot_aucA_mean, boot_aucA_ci, _ = bootstrap_auc_cluster(dfp.dropna(subset=["p_lowU_1d_mouse"]), label_A, "p_lowU_1d_mouse", B=5000, seed=20)
boot_aucB_mean, boot_aucB_ci, _ = bootstrap_auc_cluster(dfp.dropna(subset=["p_lowU_1d_mouse"]), label_B, "p_lowU_1d_mouse", B=5000, seed=21)

print("\nDiscrimination using per-mouse p_lowU_1d_mouse:")
print(f"  AUC vs day==1: {aucA:.3f} | bootstrap mean={boot_aucA_mean:.3f} 95% CI=[{boot_aucA_ci[0]:.3f}, {boot_aucA_ci[1]:.3f}]")
print(f"  AUC vs 1,5,9,13: {aucB:.3f} | bootstrap mean={boot_aucB_mean:.3f} 95% CI=[{boot_aucB_ci[0]:.3f}, {boot_aucB_ci[1]:.3f}]")

# p_lowU distribution sanity
p_desc = dfp["p_lowU_1d_mouse"].describe()
frac_high = float((dfp["p_lowU_1d_mouse"] > 0.8).mean())
frac_low  = float((dfp["p_lowU_1d_mouse"] < 0.2).mean())
p_mean_A_estrus = float(dfp.loc[dfp["day"].isin(label_A), "p_lowU_1d_mouse"].mean())
p_mean_A_non    = float(dfp.loc[~dfp["day"].isin(label_A), "p_lowU_1d_mouse"].mean())

print("\np_lowU_1d_mouse distribution:")
print(p_desc)
print(f"  frac(p_lowU_1d_mouse > 0.8) = {frac_high:.3f}")
print(f"  frac(p_lowU_1d_mouse < 0.2) = {frac_low:.3f}")
print(f"  mean p_lowU_1d_mouse (estrus day==1) = {p_mean_A_estrus:.3f}")
print(f"  mean p_lowU_1d_mouse (non-estrus)   = {p_mean_A_non:.3f}")

# Diagnostics summary
n_ok = int(diag["ok_fit"].sum())
n_bad = int((~diag["ok_fit"]).sum())
print("\nPer-mouse GMM diagnostics:")
print(f"  ok_fit mice = {n_ok}/{len(diag)} | flagged mice = {n_bad}/{len(diag)}")
print("  Flag reasons counts:")
print(diag["reason"].value_counts(dropna=False))

# Cyclicity
print("\nLag-4 cyclicity (label-free) on z_U_1_3h:")
print(f"  observed mean lag-4 autocorr = {obs:.4f}")
print(f"  Monte Carlo p-value (>= obs) = {p_mc:.4g}")
print(f"  null mean={null.mean():.4f} std={null.std():.4f}")

print("\nTop 5 mice by lowest sep (most overlapped components):")
display(diag.sort_values("sep").head())

print("="*100)